# RNN y *Backpropagation Through Time* (BPTT)

Este notebook es el complemento de [`esn.ipynb`](esn.ipynb). Usamos **la misma
tarea** —predecir la serie caótica de Mackey-Glass— pero ahora con una red
recurrente **entrenada de verdad**, para ver qué cambia.

## La idea: desenrollar la red en el tiempo

Una RNN aplica la misma función en cada paso de tiempo:

$$h_t = \tanh(W_{in}\, u_t + W_{rec}\, h_{t-1} + b), \qquad y_t = W_{out}\, h_t + b_{out}$$

El problema es que $h_t$ depende de $h_{t-1}$, que depende de $h_{t-2}$, y así
hasta el principio de la secuencia. No es una red de 3 capas: es una red de $T$
capas, donde $T$ es el largo de la secuencia, y **todas comparten los mismos
pesos**.

**BPTT es simplemente backpropagation aplicado a esa red desenrollada.** No es un
algoritmo nuevo: es la regla de la cadena recorriendo la cadena temporal hacia
atrás. Lo único particular es que, como los pesos se reusan en cada paso, el
gradiente de $W_{rec}$ es la **suma** de las contribuciones de todos los pasos:

$$\frac{\partial L}{\partial W_{rec}} = \sum_{t=1}^{T} \frac{\partial L_t}{\partial W_{rec}}$$

Y ahí aparece el problema central de las RNN. Para que el gradiente del paso $T$
llegue al paso $t$, tiene que atravesar $T-t$ multiplicaciones por la misma
matriz. Eso es aproximadamente $W_{rec}^{T-t}$, así que el gradiente **decae o
crece exponencialmente** con la distancia temporal. Lo vamos a medir en la
sección 4.

## Contra la ESN

| | ESN (`esn.ipynb`) | RNN con BPTT (este) |
|---|---|---|
| `W_rec` | aleatoria y fija | **se entrena** |
| Entrenamiento | regresión lineal, forma cerrada | gradiente descendente, muchas épocas |
| Costo | segundos | minutos |
| Problema típico | elegir bien el radio espectral | gradientes que se desvanecen o explotan |

In [1]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import time

torch.manual_seed(0)
np.random.seed(0)

---
## 1. Los datos: la misma serie de Mackey-Glass

Igual que en el notebook de ESN, para poder comparar los resultados de frente.

In [2]:
def mackey_glass(n_steps, tau=17, beta=0.2, gamma=0.1, n=10, x0=1.2, seed=0, descarte=1000):
    """Integra la ecuación de Mackey-Glass por Euler con dt=1."""
    rng = np.random.default_rng(seed)
    total = n_steps + descarte
    x = np.zeros(total + tau)
    x[:tau] = x0 + 0.01 * rng.standard_normal(tau)
    for t in range(tau, total + tau - 1):
        xt, xtau = x[t], x[t - tau]
        x[t + 1] = xt + beta * xtau / (1 + xtau**n) - gamma * xt
    return x[tau + descarte:]

serie = mackey_glass(6000)
s = (serie - serie.mean()) / serie.std()

U = torch.tensor(s[:-1], dtype=torch.float32).view(-1, 1)   # entrada  u(n) = x(n)
D = torch.tensor(s[1:],  dtype=torch.float32).view(-1, 1)   # objetivo d(n) = x(n+1)

U_tr, D_tr = U[:3000], D[:3000]
U_te, D_te = U[3000:5000], D[3000:5000]
WASHOUT = 200

def nrmse(y, d):
    y = y.detach().numpy() if torch.is_tensor(y) else y
    d = d.detach().numpy() if torch.is_tensor(d) else d
    return float(np.sqrt(np.mean((y - d)**2) / np.var(d)))

print("entrenamiento:", tuple(U_tr.shape), "| test:", tuple(U_te.shape))

entrenamiento: (3000, 1) | test: (2000, 1)


---
## 2. La RNN, con la recurrencia escrita a mano

Podríamos usar `nn.RNN`, que hace todo en C++ y es mucho más rápido. Pero acá el
objetivo es **ver** el BPTT, así que escribimos el bucle temporal explícito en
Python. Cada iteración agrega un eslabón más a la cadena que después hay que
recorrer hacia atrás.

Fijate que no hay nada especial en el `forward`: es un `for` que reusa las mismas
tres capas. Autograd se encarga solo de armar el grafo desenrollado.

In [3]:
class RNNManual(nn.Module):
    """RNN 'vanilla' con el bucle temporal explícito."""

    def __init__(self, n_in=1, n_hidden=64, n_out=1):
        super().__init__()
        self.n_hidden = n_hidden
        self.W_in  = nn.Linear(n_in, n_hidden, bias=False)
        self.W_rec = nn.Linear(n_hidden, n_hidden, bias=True)
        self.W_out = nn.Linear(n_hidden, n_out)

    def forward(self, u_seq, h0=None, guardar_estados=False):
        h = torch.zeros(self.n_hidden) if h0 is None else h0
        salidas, estados = [], []
        for t in range(u_seq.shape[0]):                       # <-- el desenrollado
            h = torch.tanh(self.W_in(u_seq[t]) + self.W_rec(h))
            if guardar_estados:
                h.retain_grad()          # para poder mirar dL/dh_t después
                estados.append(h)
            salidas.append(self.W_out(h))
        Y = torch.stack(salidas)
        return (Y, h, estados) if guardar_estados else (Y, h)

modelo = RNNManual(n_hidden=64)
print("parámetros entrenables:", sum(p.numel() for p in modelo.parameters()))
for nombre, p in modelo.named_parameters():
    print(f"  {nombre:12s} {tuple(p.shape)}")

parámetros entrenables: 4289
  W_in.weight  (64, 1)
  W_rec.weight (64, 64)
  W_rec.bias   (64,)
  W_out.weight (1, 64)
  W_out.bias   (1,)


### Ver el grafo desenrollado

Autograd construye una cadena de operaciones, un eslabón por paso de tiempo.
Podemos recorrerla hacia atrás desde la salida y contar los nodos: la profundidad
del grafo crece linealmente con el largo de la secuencia. **Eso es lo que hace
caro al BPTT en memoria**: hay que guardar las activaciones de los $T$ pasos hasta
que termine el `backward()`.

In [ ]:
def profundidad_grafo(tensor):
    """Cuenta los nodos del grafo de autograd colgando de un tensor."""
    vistos, pila, n = set(), [tensor.grad_fn], 0
    while pila:
        fn = pila.pop()
        if fn is None or fn in vistos:
            continue
        vistos.add(fn); n += 1
        pila.extend(sig for sig, _ in fn.next_functions)
    return n

for T in [1, 5, 10, 50, 100]:
    Y, _ = modelo(U_tr[:T])
    print(f"secuencia de {T:3d} pasos -> {profundidad_grafo(Y.sum()):5d} nodos en el grafo")

---
## 3. Entrenar con BPTT truncado

Propagar el gradiente por los 3000 pasos de la serie sería carísimo (y con
gradientes inútiles, como vamos a ver). Lo que se hace en la práctica es **BPTT
truncado**: cortar la secuencia en ventanas de `T_bptt` pasos y propagar solo
dentro de cada ventana.

El detalle clave está en esta línea:

```python
h = h.detach()
```

El estado `h` **se arrastra** de una ventana a la siguiente, así que la red nunca
pierde el contexto. Pero `detach()` corta el grafo en ese punto, así que el
gradiente **no** retrocede más allá de la ventana actual. Se separan dos cosas que
uno tiende a confundir: cuánta memoria tiene la red hacia adelante, y cuánto
retrocede el gradiente hacia atrás.

In [ ]:
def entrenar_bptt(model, U, D, T_bptt=100, epocas=30, lr=3e-3, clip=1.0, registrar=False):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist, normas = [], []

    for ep in range(epocas):
        h = torch.zeros(model.n_hidden)      # estado inicial de la época
        total, n_vent = 0.0, 0

        for ini in range(0, len(U) - T_bptt, T_bptt):
            opt.zero_grad()
            Y, h = model(U[ini:ini + T_bptt], h0=h)
            loss = ((Y - D[ini:ini + T_bptt])**2).mean()
            loss.backward()                                   # <-- BPTT

            if registrar:                                     # norma ANTES de recortar
                normas.append(torch.sqrt(sum(p.grad.pow(2).sum()
                                             for p in model.parameters())).item())
            if clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()

            h = h.detach()        # <-- el estado sigue, el gradiente no
            total += loss.item(); n_vent += 1

        hist.append(total / n_vent)
        if ep % 5 == 0 or ep == epocas - 1:
            print(f"  época {ep:3d}: loss {hist[-1]:.6f}")

    return (hist, normas) if registrar else hist

In [ ]:
torch.manual_seed(0)
modelo = RNNManual(n_hidden=64)

t0 = time.time()
hist, normas = entrenar_bptt(modelo, U_tr, D_tr, T_bptt=100, epocas=30, registrar=True)
t_rnn = time.time() - t0

with torch.no_grad():
    Y_te, _ = modelo(U_te)
err_rnn = nrmse(Y_te[WASHOUT:], D_te[WASHOUT:])

print(f"\ntiempo de entrenamiento: {t_rnn:.1f} s")
print(f"NRMSE en test: {err_rnn:.5f}")

### Comparación directa con la ESN

Corremos una ESN mínima sobre exactamente los mismos datos. Es la misma idea del
otro notebook, condensada en unas pocas líneas para que este sea autónomo.

In [ ]:
def esn_rapida(U_tr, D_tr, U_te, n_res=400, radio=0.95, sparsity=0.1,
               ridge=1e-7, washout=200, seed=42):
    """ESN mínima: reservorio aleatorio fijo + lectura lineal por ridge."""
    rng = np.random.default_rng(seed)
    u_tr = np.column_stack([U_tr.numpy().ravel(), np.ones(len(U_tr))])   # con bias
    u_te = np.column_stack([U_te.numpy().ravel(), np.ones(len(U_te))])

    W_in = rng.uniform(-1, 1, (n_res, 2))
    W = rng.uniform(-1, 1, (n_res, n_res)) * (rng.random((n_res, n_res)) < sparsity)
    W *= radio / np.max(np.abs(np.linalg.eigvals(W)))

    def estados(u):
        X = np.zeros((len(u), n_res)); x = np.zeros(n_res)
        for i in range(len(u)):
            x = np.tanh(W_in @ u[i] + W @ x)
            X[i] = x
        return X

    X = estados(u_tr)[washout:]
    d = D_tr.numpy()[washout:]
    W_out = d.T @ X @ np.linalg.inv(X.T @ X + ridge * np.eye(n_res))
    return estados(u_te) @ W_out.T

t0 = time.time()
Y_esn = esn_rapida(U_tr, D_tr, U_te)
t_esn = time.time() - t0
err_esn = nrmse(Y_esn[WASHOUT:], D_te[WASHOUT:].numpy())

print(f"{'':<22} {'NRMSE':>10} {'tiempo':>10} {'params entrenados':>20}")
print(f"{'RNN + BPTT':<22} {err_rnn:10.5f} {t_rnn:9.1f}s "
      f"{sum(p.numel() for p in modelo.parameters()):20d}")
print(f"{'ESN (reservorio)':<22} {err_esn:10.5f} {t_esn:9.1f}s {400:20d}")

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].semilogy(hist)
axs[0].set_xlabel("época"); axs[0].set_ylabel("MSE de entrenamiento")
axs[0].set_title("la RNN necesita épocas; la ESN no tiene curva"); axs[0].grid(alpha=0.3)

ini = WASHOUT
axs[1].plot(D_te[ini:ini+250], lw=2, label="real")
axs[1].plot(Y_te[ini:ini+250], "--", label=f"RNN (NRMSE {err_rnn:.4f})")
axs[1].plot(Y_esn[ini:ini+250], ":", lw=2, label=f"ESN (NRMSE {err_esn:.4f})")
axs[1].set_xlabel("paso de test"); axs[1].legend(); axs[1].grid(alpha=0.3)
axs[1].set_title("predicción a un paso")
plt.tight_layout(); plt.show()

En esta tarea la ESN gana claramente: cerca de 20 veces menos error, y su
entrenamiento pasa de decenas de segundos a una fracción de segundo. No es un
resultado universal —Mackey-Glass es exactamente el
tipo de problema para el que las ESN fueron diseñadas— pero ilustra bien el
argumento del *reservoir computing*: si un reservorio aleatorio ya genera las
dinámicas que necesitás, entrenar la recurrencia es esfuerzo desperdiciado.

Lo que sí conviene notar es el tamaño. La RNN llega a ese error con **4289
parámetros en total**, mientras que la ESN necesita un reservorio de 161.200 pesos
(aunque entrene solo 400 de ellos). La recurrencia entrenada es mucho más
eficiente *por parámetro*, porque se adapta al problema en vez de esperar que el
azar genere las dinámicas útiles. Cuando la tarea es compleja y los datos
abundan, esa eficiencia es la que termina ganando: por eso el mundo entrena
LSTMs y no reservorios.

---
## 4. El problema central: el gradiente a través del tiempo

Este es el experimento más importante del notebook.

Corremos una secuencia de 100 pasos, ponemos un "error" solo al **final**, y
medimos cuánto vale $\|\partial L / \partial h_t\|$ para cada $t$ hacia atrás. Es
decir: **cuánta información sobre el error final llega a cada paso pasado**.

Repetimos con `W_rec` escalada a distintos tamaños. La escala hace de análogo del
radio espectral del notebook de ESN, y el resultado es el espejo exacto de aquel:

In [ ]:
def medir_gradiente(escala, T=100, n_hidden=64, seed=0):
    """||dL/dh_t|| para cada t, con un error puesto solo al final de la secuencia."""
    g = torch.Generator().manual_seed(seed)
    W_rec = (torch.randn(n_hidden, n_hidden, generator=g) / np.sqrt(n_hidden)) * escala
    W_in  = torch.randn(n_hidden, 1, generator=g) * 0.1
    W_rec.requires_grad_(True)

    u = torch.randn(T, 1, generator=g) * 0.1
    h, hs = torch.zeros(n_hidden), []
    for t in range(T):
        h = torch.tanh(W_in @ u[t] + W_rec @ h)
        h.retain_grad()
        hs.append(h)

    hs[-1].pow(2).sum().backward()          # el error entra solo en el último paso
    return np.array([e.grad.norm().item() for e in hs])

escalas = [0.5, 0.9, 1.0, 1.2, 2.0]
plt.figure(figsize=(9, 5))
for esc in escalas:
    g = medir_gradiente(esc)
    pasos_atras = np.arange(len(g))[::-1]           # 0 = el paso final
    plt.semilogy(pasos_atras, np.maximum(g, 1e-30), label=f"escala {esc}")
    print(f"escala {esc:4.1f}: en t=T {g[-1]:9.2e} | 50 pasos atrás {g[-51]:9.2e} "
          f"| 100 atrás {g[0]:9.2e}")

plt.gca().invert_xaxis()
plt.xlabel("pasos hacia atrás desde el final de la secuencia")
plt.ylabel(r"$\|\partial L / \partial h_t\|$")
plt.title("cuánto gradiente llega hacia atrás en el tiempo")
plt.legend(); plt.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

### Cómo leer esto

El eje vertical es logarítmico, así que las rectas son **decaimientos o
crecimientos exponenciales**:

- **Escala 0.5**: el gradiente cae unos 16 órdenes de magnitud en 50 pasos y a los
  100 llega a cero exacto por *underflow* de punto flotante. La red es literalmente
  incapaz de aprender cualquier dependencia más larga que unos pocos pasos: la
  señal de error no llega. Esto es el **gradiente que se desvanece**.
- **Escala 1.0**: el gradiente se mantiene aproximadamente constante. Es el único
  régimen donde el aprendizaje de largo plazo es posible, y es un equilibrio
  frágil.
- **Escala 2.0**: el gradiente crece 7 órdenes de magnitud. Un solo paso de
  descenso manda los pesos al infinito y la pérdida se vuelve `NaN`. Es el
  **gradiente que explota**.

Es la misma matemática que el radio espectral de la ESN, vista desde el otro lado.
Allá el radio controlaba cuánto **recuerda** el reservorio hacia adelante; acá
controla cuánto **aprende** la red hacia atrás. En ambos casos el punto crítico
está en 1, y en ambos casos alejarse arruina todo.

La diferencia es qué se puede hacer al respecto. La ESN fija el radio a mano de
una vez y listo. La RNN **entrena** `W_rec`, así que su escala cambia durante el
entrenamiento y puede caerse a cualquiera de los dos regímenes malos sin que uno
se entere. De ahí vienen los dos parches estándar:

- **Recorte de gradiente** (*gradient clipping*) para la explosión — la línea
  `clip_grad_norm_` de nuestra función de entrenamiento.
- **LSTM y GRU** para el desvanecimiento: agregan un camino aditivo para el estado
  (la celda de memoria) por el que el gradiente fluye sin multiplicarse por
  `W_rec` en cada paso.

### El clipping, en la práctica

Durante el entrenamiento de la sección 3 fuimos guardando la norma del gradiente
**antes** de recortarla. Vale la pena mirar la distribución.

In [ ]:
normas = np.array(normas)
print(f"norma del gradiente: mediana {np.median(normas):.3f} | "
      f"percentil 99 {np.percentile(normas, 99):.3f} | máximo {normas.max():.3f}")
print(f"pasos recortados por clip=1.0: {(normas > 1.0).sum()} de {len(normas)} "
      f"({100*(normas > 1.0).mean():.1f}%)")

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].semilogy(normas, lw=0.5)
axs[0].axhline(1.0, color="tab:red", ls="--", label="umbral de clip")
axs[0].set_xlabel("paso de optimización"); axs[0].set_ylabel("norma del gradiente")
axs[0].set_title("los picos son el gradiente explotando"); axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].hist(np.log10(normas), bins=60)
axs[1].axvline(0, color="tab:red", ls="--", label="clip = 1.0")
axs[1].set_xlabel(r"$\log_{10}$ de la norma"); axs[1].set_ylabel("frecuencia")
axs[1].set_title("distribución"); axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

La distribución tiene **cola larga**: la enorme mayoría de los pasos tienen un
gradiente chico y de vez en cuando aparece uno órdenes de magnitud más grande. Sin
recorte, uno solo de esos picos alcanza para tirar los pesos lejos y destruir el
entrenamiento acumulado. El clipping preserva la **dirección** del gradiente y
solo le limita el **módulo**, así que en los pasos normales no hace absolutamente
nada.

---
## 5. ¿Cuánto hay que retroceder?

`T_bptt` controla hasta dónde llega el gradiente hacia atrás. ¿Cuánto importa?

Mackey-Glass no sirve para contestar esto: la serie es suave, y para predecir el
próximo valor alcanza casi con mirar los últimos pasos, así que truncar poco o
mucho da parecido. Necesitamos una tarea donde la memoria requerida esté
**declarada explícitamente**.

Usamos la **tarea de retardo**: la entrada es ruido blanco y la salida tiene que
reproducirlo $k$ pasos después.

$$u(t) \sim \mathcal{N}(0,1), \qquad d(t) = u(t-k)$$

Con $k = 15$ la red está obligada a guardar un valor durante 15 pasos y devolverlo
intacto. No hay ningún atajo posible: como la entrada es ruido blanco, el valor no
se puede inferir de nada más.

**Nota metodológica**: mantenemos **fijo el número de pasos del optimizador**
(2000) para todos los truncamientos. Si fijáramos las épocas, los truncamientos
cortos harían muchas más actualizaciones y la comparación estaría midiendo dos
cosas a la vez.

In [ ]:
def datos_retardo(n_batch, L, k, seed):
    """Ruido blanco de entrada; la salida es la entrada retrasada k pasos."""
    g = torch.Generator().manual_seed(seed)
    u = torch.randn(n_batch, L, 1, generator=g)
    d = torch.zeros(n_batch, L, 1)
    d[:, k:] = u[:, :-k]
    return u, d

class RNNsimple(nn.Module):
    """Acá sí usamos nn.RNN: es la misma recurrencia, pero mucho más rápida."""
    def __init__(self, n_hidden=48):
        super().__init__()
        self.rnn = nn.RNN(1, n_hidden, batch_first=True)
        self.fc  = nn.Linear(n_hidden, 1)
    def forward(self, x, h0=None):
        h, hn = self.rnn(x, h0)
        return self.fc(h), hn

def entrenar_retardo(T_bptt, k, L=96, n_batch=32, updates=2000, lr=3e-3, seed=0):
    torch.manual_seed(seed)
    m = RNNsimple()
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    hechos, semilla = 0, 0

    while hechos < updates:
        u, d = datos_retardo(n_batch, L, k, seed=semilla); semilla += 1
        h = None
        for ini in range(0, L - T_bptt + 1, T_bptt):
            opt.zero_grad()
            y, h = m(u[:, ini:ini + T_bptt], h)
            loss = ((y - d[:, ini:ini + T_bptt])**2).mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
            opt.step()
            h = h.detach()          # el estado sigue, el gradiente se corta acá
            hechos += 1
            if hechos >= updates:
                break

    u, d = datos_retardo(16, L, k, seed=9999)          # secuencias nuevas
    with torch.no_grad():
        y, _ = m(u)
    err = ((y[:, k+5:] - d[:, k+5:])**2).mean().item()
    return float(np.sqrt(err / d[:, k+5:].var().item()))

In [ ]:
K = 15
truncamientos = [1, 2, 4, 8, 12, 16, 24, 32, 48, 96]
err_trunc = []

print(f"tarea: d(t) = u(t-{K}) | 2000 pasos del optimizador en todos los casos\n")
print(f"{'T_bptt':>7} {'NRMSE':>9}")
for T in truncamientos:
    err_trunc.append(entrenar_retardo(T, K))
    print(f"{T:7d} {err_trunc[-1]:9.4f}")

plt.figure(figsize=(8, 4.5))
plt.semilogx(truncamientos, err_trunc, marker="o")
plt.axvline(K, color="tab:red", ls="--", label=f"k = {K} (memoria que exige la tarea)")
plt.axhline(1.0, color="gray", ls=":", label="NRMSE = 1: no aprendió nada")
plt.xlabel("T_bptt (pasos que retrocede el gradiente)")
plt.ylabel("NRMSE")
plt.title(f"la red solo aprende si el gradiente llega más atrás que k={K}")
plt.legend(); plt.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

### El resultado

La transición es casi binaria:

- Con `T_bptt` ≤ 16 el NRMSE es **1.0**, que es exactamente lo que da predecir
  siempre la media. La red no aprendió absolutamente nada.
- Con `T_bptt` = 24 el error se desploma a ~0.10, y sigue mejorando de ahí en
  adelante.

El salto cae justo por encima de $k=15$, la memoria que la tarea exige. Que el
umbral esté algo más arriba y no exactamente en 15 tiene sentido: las ventanas se
cortan en posiciones fijas, así que un valor que entra cerca del final de una
ventana queda separado de su salida por el borde. Hace falta una ventana bastante
más larga que $k$ para que la mayoría de los pares entrada-salida caigan **dentro**
de la misma ventana de gradiente.

Lo importante conceptualmente: en todos estos casos la red **sí tiene** su estado
recurrente hacia adelante —lo arrastramos con `detach` entre ventanas— así que
físicamente podría recordar. Lo que le falta con truncamientos cortos es la señal
de aprendizaje: nunca se entera de que el valor que vio hace 15 pasos era el que
había que guardar. **Recordar y aprender a recordar son dos cosas distintas, y
BPTT es lo que hace posible la segunda.**

Y de ahí se entiende por qué el desvanecimiento del gradiente es tan grave: no
hace falta truncar a mano para quedarse sin gradiente. Si `W_rec` cae en el
régimen contractivo de la sección 4, el gradiente se desvanece **solo** antes de
llegar, con el mismo efecto que truncar, aunque uno crea que está propagando por
100 pasos.

---
## 6. Resumen

**Qué es BPTT**: backpropagation aplicado a la red desenrollada en el tiempo. La
única particularidad es que los pesos se comparten entre pasos, así que sus
gradientes se suman a lo largo de toda la secuencia.

**Por qué es difícil**: para llegar $k$ pasos atrás, el gradiente se multiplica $k$
veces por (aproximadamente) la misma matriz. El resultado decae o crece de forma
exponencial. Casi nunca se queda quieto.

**Los parches**:

| problema | síntoma | solución |
|---|---|---|
| gradiente que explota | pérdida a `NaN`, saltos bruscos | recorte de la norma (`clip_grad_norm_`) |
| gradiente que se desvanece | no aprende dependencias largas | LSTM / GRU, o inicialización cerca de 1 |
| costo en memoria | se guardan $T$ activaciones | BPTT truncado |

**Contra la ESN**: son dos respuestas opuestas al mismo problema. La ESN lo
**esquiva** —no entrena la recurrencia, así que no hay gradiente que propagar— a
costa de necesitar un reservorio grande y de depender de que el azar genere las
dinámicas útiles. La RNN lo **enfrenta**, y a cambio adapta la recurrencia al
problema y es mucho más eficiente por parámetro.

### Para seguir jugando

- Cambiar `clip=None` en el entrenamiento de la sección 3 y ver si la pérdida se
  desestabiliza.
- Reemplazar `torch.tanh` por `torch.relu` en `RNNManual`: sin la saturación de la
  `tanh` que contiene el estado, la explosión es mucho más fácil.
- Sustituir la recurrencia por `nn.LSTM` y comparar en la misma tarea.
- Inicializar `W_rec` como la identidad (`nn.init.eye_`), que es el truco de la
  *IRNN*: arranca justo en el régimen donde el gradiente no se desvanece.